# Convert CSV to Parquet

In [28]:
import pandas as pd

## Download weather data

First, we need to download the data.

sample download link for Sydney Airport:
https://www.bom.gov.au/climate/dwo/202605/text/IDCJDW2125.202605.csv
https://www.bom.gov.au/climate/dwo/202605/text/IDCJDW2119.202605.csv

In [29]:
base_download_link = 'https://www.bom.gov.au/climate/dwo/'
dates = ['202605', '202604', '202603', '202602', '202601',
        '202512', '202511', '202510', '202509', '202508',
        '202507', '202506', '202505', '202504', '202503',
        '202502', '202501']
locations = ['IDCJDW2125', 'IDCJDW2119']

def get_download_link(location, date):
    """
    download the data for a specific location and date.
    """
    return base_download_link + date + '/text/' + location + '.' + date + '.csv'

def download(out_dir):
    """
    Download weather data CSV files for all locations and dates.

    Args:
        out_dir: Output directory to save the downloaded files
    """
    import os
    import requests
    import time

    os.makedirs(out_dir, exist_ok=True)

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'max-age=0'
    }

    for location in locations:
        for date in dates:
            url = get_download_link(location, date)
            filename = f"{location}.{date}.csv"
            filepath = os.path.join(out_dir, filename)

            try:
                response = requests.get(url, headers=headers, timeout=30)
                if response.status_code == 200:
                    with open(filepath, 'wb') as f:
                        f.write(response.content)
                    print(f"Downloaded: {filename}")
                else:
                    print(f"Failed to download {filename}: HTTP {response.status_code}")
            except Exception as e:
                print(f"Error downloading {filename}: {e}")

            time.sleep(0.5)

    print("Download complete!")


In [30]:
download("./weather_data")

Downloaded: IDCJDW2125.202605.csv
Downloaded: IDCJDW2125.202604.csv
Downloaded: IDCJDW2125.202603.csv
Downloaded: IDCJDW2125.202602.csv
Downloaded: IDCJDW2125.202601.csv
Downloaded: IDCJDW2125.202512.csv
Downloaded: IDCJDW2125.202511.csv
Downloaded: IDCJDW2125.202510.csv
Downloaded: IDCJDW2125.202509.csv
Downloaded: IDCJDW2125.202508.csv
Downloaded: IDCJDW2125.202507.csv
Downloaded: IDCJDW2125.202506.csv
Downloaded: IDCJDW2125.202505.csv
Downloaded: IDCJDW2125.202504.csv
Downloaded: IDCJDW2125.202503.csv
Failed to download IDCJDW2125.202502.csv: HTTP 404
Failed to download IDCJDW2125.202501.csv: HTTP 404
Downloaded: IDCJDW2119.202605.csv
Downloaded: IDCJDW2119.202604.csv
Downloaded: IDCJDW2119.202603.csv
Downloaded: IDCJDW2119.202602.csv
Downloaded: IDCJDW2119.202601.csv
Downloaded: IDCJDW2119.202512.csv
Downloaded: IDCJDW2119.202511.csv
Downloaded: IDCJDW2119.202510.csv
Downloaded: IDCJDW2119.202509.csv
Downloaded: IDCJDW2119.202508.csv
Downloaded: IDCJDW2119.202507.csv
Downloaded: ID

## Combine data to parquet file

Now, we can read the csv files and combine them into a single parquet file.

In [31]:
def combine_data(csv_dir="./weather_data", output_file="./weather_data.parquet"):
    """
    Read all CSV files from the directory and combine them into a single parquet file.
    
    Args:
        csv_dir: Directory containing the CSV files
        output_file: Path to save the combined parquet file
    """
    import os
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    
    csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
    
    if not csv_files:
        print(f"No CSV files found in {csv_dir}")
        return
    
    def find_header_row(filepath):
        """Find the row number where the actual data header starts."""
        with open(filepath, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if line.startswith(',"Date"'):
                    return i
        return None
    
    dataframes = []
    for csv_file in csv_files:
        filepath = os.path.join(csv_dir, csv_file)
        try:
            header_row = find_header_row(filepath)
            if header_row is None:
                print(f"Could not find header in {csv_file}")
                continue
            df = pd.read_csv(filepath, encoding='latin-1', skiprows=header_row, low_memory=False)
            df['source_file'] = csv_file
            dataframes.append(df)
            print(f"Loaded: {csv_file}")
        except Exception as e:
            print(f"Error loading {csv_file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        
        for col in combined_df.columns:
            if combined_df[col].dtype == 'object':
                combined_df[col] = combined_df[col].astype(str)
        
        table = pa.Table.from_pandas(combined_df)
        pq.write_table(table, output_file)
        
        print(f"Combined {len(dataframes)} files into {output_file}")
        print(f"Total rows: {len(combined_df)}")
        return combined_df
    else:
        print("No data to save")
        return None

In [32]:
combine_data()

Loaded: IDCJDW2125.202605.csv
Loaded: IDCJDW2125.202604.csv
Loaded: IDCJDW2125.202603.csv
Loaded: IDCJDW2125.202602.csv
Loaded: IDCJDW2125.202601.csv
Loaded: IDCJDW2125.202504.csv
Loaded: IDCJDW2125.202510.csv
Loaded: IDCJDW2125.202511.csv
Loaded: IDCJDW2125.202505.csv
Loaded: IDCJDW2125.202507.csv
Loaded: IDCJDW2125.202506.csv
Loaded: IDCJDW2125.202512.csv
Loaded: IDCJDW2125.202503.csv
Loaded: IDCJDW2119.202509.csv
Loaded: IDCJDW2119.202508.csv
Loaded: IDCJDW2119.202505.csv
Loaded: IDCJDW2119.202511.csv
Loaded: IDCJDW2119.202510.csv
Loaded: IDCJDW2119.202504.csv
Loaded: IDCJDW2119.202512.csv
Loaded: IDCJDW2119.202506.csv
Loaded: IDCJDW2119.202507.csv
Loaded: IDCJDW2119.202503.csv
Loaded: IDCJDW2125.202508.csv
Loaded: IDCJDW2125.202509.csv
Loaded: IDCJDW2119.202604.csv
Loaded: IDCJDW2119.202605.csv
Loaded: IDCJDW2119.202602.csv
Loaded: IDCJDW2119.202603.csv
Loaded: IDCJDW2119.202601.csv
Combined 30 files into ./weather_data.parquet
Total rows: 874


,Unnamed: 0,Date,Minimum temperature (°C),Maximum temperature (°C),Rainfall (mm),Evaporation (mm),Sunshine (hours),Direction of maximum wind gust,Speed of maximum wind gust (km/h),Time of maximum wind gust,...,9am wind direction,9am wind speed (km/h),9am MSL pressure (hPa),3pm Temperature (°C),3pm relative humidity (%),3pm cloud amount (oktas),3pm wind direction,3pm wind speed (km/h),3pm MSL pressure (hPa),source_file
0,NaN,2026-05-1,15.6,23.7,0.2,NaN,NaN,ENE,28.0,22:33,...,WNW,15,1030.2,22.9,54.0,5.0,ENE,17.0,1026.9,IDCJDW2125.202605.csv
1,NaN,2026-05-2,17.1,25.4,0.0,7.2,5.6,NNE,50.0,13:51,...,W,9,1026.2,23.9,62.0,7.0,NNE,28.0,1022.9,IDCJDW2125.202605.csv
2,NaN,2026-05-3,17.2,25.6,0.0,4.0,7.3,NNE,48.0,19:46,...,N,20,1022.8,24.7,57.0,7.0,NE,33.0,1018.7,IDCJDW2125.202605.csv
3,NaN,2026-05-4,18.1,22.1,0.0,4.8,1.5,N,35.0,19:28,...,NW,15,1017.8,21.0,72.0,7.0,NNE,22.0,1014.5,IDCJDW2125.202605.csv
4,NaN,2026-05-5,14.4,23.7,5.2,2.8,10.2,W,48.0,10:32,...,WNW,28,1019.2,22.3,43.0,1.0,SE,17.0,1017.1,IDCJDW2125.202605.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869,NaN,2026-01-27,18.9,29.8,0.0,NaN,NaN,E,31.0,18:12,...,SE,6,1015.4,28.9,47.0,1.0,ENE,17,1011.8,IDCJDW2119.202601.csv
870,NaN,2026-01-28,15.0,35.1,0.0,NaN,NaN,ENE,26.0,12:34,...,NNE,9,1013.5,32.1,35.0,NaN,SE,6,1010.3,IDCJDW2119.202601.csv
871,NaN,2026-01-29,20.5,32.0,0.0,NaN,NaN,ESE,31.0,15:56,...,NNW,4,1019.2,30.9,44.0,NaN,SSE,9,1015.9,IDCJDW2119.202601.csv
872,NaN,2026-01-30,18.7,34.3,0.0,NaN,NaN,E,28.0,18:45,...,NNE,9,1017.9,32.0,43.0,NaN,NE,13,1012.2,IDCJDW2119.202601.csv
